In [ ]:
# ============================================================
# CELL 1: Environment + Observability + Budget + OOM Guards
# ============================================================
import os, sys, time, gc, warnings, math, random, zipfile
warnings.filterwarnings("ignore")

# ----------------------------
# Runtime budget (leave buffer)
# ----------------------------
RUN_START = time.monotonic()

# Kaggle GPU notebook hard cap is 9h. Leave ~10–15 min buffer for zip + overhead.
MAX_RUNTIME_SEC = int(8.75 * 3600)  # 8h45m budget (safer than 9h)

def elapsed_s() -> float:
    return time.monotonic() - RUN_START

def elapsed_h() -> float:
    return elapsed_s() / 3600.0

def remaining_s() -> float:
    return MAX_RUNTIME_SEC - elapsed_s()

def remaining_h() -> float:
    return remaining_s() / 3600.0

def hard_budget_check(tag: str = "", min_remaining_min: int = 10) -> None:
    """Fail-fast if we are too close to runtime limit."""
    rem = remaining_s()
    print(f"[TIME] {tag} | elapsed={elapsed_h():.2f}h | remaining={max(rem,0)/3600.0:.2f}h")
    if rem < min_remaining_min * 60:
        raise RuntimeError(f"[BUDGET] Too close to runtime limit (<{min_remaining_min} min). Abort to avoid Kaggle kill.")

hard_budget_check("Pipeline start")

# ----------------------------
# Memory / GPU telemetry helpers
# ----------------------------
def _try_import_psutil():
    try:
        import psutil  # type: ignore
        return psutil
    except Exception:
        return None

_PSUTIL = _try_import_psutil()

def ram_gb() -> float:
    try:
        if _PSUTIL:
            return _PSUTIL.Process(os.getpid()).memory_info().rss / (1024**3)
    except Exception:
        pass
    # Fallback: unknown
    return float("nan")

def gpu_mem_gb():
    """Returns (allocated_GB, reserved_GB, peak_allocated_GB) or (nan,nan,nan) if unavailable."""
    try:
        import torch
        if not torch.cuda.is_available():
            return (float("nan"), float("nan"), float("nan"))
        alloc = torch.cuda.memory_allocated() / (1024**3)
        reserv = torch.cuda.memory_reserved() / (1024**3)
        peak = torch.cuda.max_memory_allocated() / (1024**3)
        return (alloc, reserv, peak)
    except Exception:
        return (float("nan"), float("nan"), float("nan"))

def log_resources(tag: str = "") -> None:
    a, r, p = gpu_mem_gb()
    rg = ram_gb()
    print(f"[RES] {tag} | RAM={rg:.2f}GB | GPU alloc={a:.2f}GB reserv={r:.2f}GB peak={p:.2f}GB")

# ----------------------------
# Heartbeat + hangtime detector
# ----------------------------
class Heartbeat:
    def __init__(self, every_s: int = 75):
        self.every_s = every_s
        self._last = time.monotonic()
        self._last_progress = time.monotonic()

    def progress(self):
        """Call this when meaningful progress happens (e.g., completed a tile/volume)."""
        self._last_progress = time.monotonic()

    def beat(self, stage: str, extra: str = ""):
        """Wall-clock heartbeat (prints at most every `every_s` seconds)."""
        now = time.monotonic()
        if now - self._last >= self.every_s:
            self._last = now
            hard_budget_check(f"HB {stage}", min_remaining_min=10)
            log_resources(f"HB {stage} {extra}".strip())

    def hang_warn(self, stage: str, warn_after_s: int = 300):
        """Warn if no progress for too long (does not stop run)."""
        now = time.monotonic()
        if now - self._last_progress >= warn_after_s:
            print(f"[HANG?] No progress for {int(now-self._last_progress)}s during '{stage}'.")
            log_resources(f"HANG {stage}")
            # Do not reset; continued warnings are fine.

HB = Heartbeat(every_s=75)

# ----------------------------
# OOM helpers (fail-fast by default)
# ----------------------------
def is_cuda_oom(err: Exception) -> bool:
    msg = str(err).lower()
    return ("out of memory" in msg) or ("cuda" in msg and "memory" in msg)

def oom_failfast(context: str, err: Exception) -> None:
    print(f"[OOM] FAIL-FAST in {context}: {err}")
    log_resources(f"OOM {context}")
    # Try to free cache to aid debugging logs, then raise.
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass
    raise RuntimeError(f"CUDA OOM in {context}. Aborting to avoid silent degraded submission.") from err

print("[OK] Observability + budget guards ready")

# ----------------------------
# TIFF loading (NO pip installs)
# ----------------------------
try:
    import tifffile  # should exist in Kaggle images
except ImportError as e:
    raise ImportError(
        "tifffile is missing. Kaggle submissions run with internet disabled, so do NOT pip install. "
        "Add tifffile to your Kaggle dataset/image or use only PIL fallback if you accept it."
    ) from e

HAS_IMAGECODECS = False
try:
    import imagecodecs  # optional accelerator
    HAS_IMAGECODECS = True
    print("[OK] imagecodecs available")
except Exception:
    print("[INFO] imagecodecs not installed — tifffile will still work, just slower for some compressions")

from PIL import Image
import numpy as np

def read_tif_volume(path: str) -> np.ndarray:
    # Prefer tifffile always; it handles multi-page TIFFs robustly.
    try:
        return tifffile.imread(path)
    except Exception as e:
        print(f"[WARN] tifffile failed on {path} ({e}). Falling back to PIL frame read (slower).")
        img = Image.open(path)
        frames = []
        try:
            for i in range(getattr(img, "n_frames", 1)):
                img.seek(i)
                frames.append(np.array(img))
        finally:
            img.close()
        return np.stack(frames, axis=0)

print("[OK] Environment ready")
log_resources("Init")

In [ ]:
# ============================================================
# CELL 2: Imports & Configuration (TOPO/VOI SCORE-SAFE)
# ============================================================
import numpy as np
import pandas as pd
import scipy.ndimage as ndi
from skimage.morphology import remove_small_objects

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import autocast

# ---- Device / determinism / perf knobs ----
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[OK] PyTorch {torch.__version__}  device={DEVICE}")

if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = True  # inference speed
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"     GPU {i}: {props.name}  VRAM: {props.total_memory / 1e9:.1f} GB")

# Autocast dtype policy (T4 supports fp16; bf16 may be slower/unavailable)
AMP_DTYPE = torch.float16 if DEVICE.type == "cuda" else torch.bfloat16

# ---- Seeds ----
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)

log_resources("After torch init")
hard_budget_check("Config start", min_remaining_min=10)

# ---- Paths (auto-discover ROOT_DIR) ----
_ROOT_CANDIDATES = [
    "/kaggle/input/competitions/vesuvius-challenge-surface-detection",
    "/kaggle/input/vesuvius-challenge-surface-detection",
]
ROOT_DIR = None
for _rc in _ROOT_CANDIDATES:
    if os.path.isdir(_rc) and os.path.exists(os.path.join(_rc, "test.csv")):
        ROOT_DIR = _rc
        break

if ROOT_DIR is None:
    _input = "/kaggle/input"
    if os.path.isdir(_input):
        print(f"[DISCOVER] /kaggle/input: {os.listdir(_input)}")
        for d in os.listdir(_input):
            fp = os.path.join(_input, d)
            if os.path.isdir(fp) and os.path.exists(os.path.join(fp, "test.csv")):
                ROOT_DIR = fp
                break
            if os.path.isdir(fp):
                for dd in os.listdir(fp):
                    fp2 = os.path.join(fp, dd)
                    if os.path.isdir(fp2) and os.path.exists(os.path.join(fp2, "test.csv")):
                        ROOT_DIR = fp2
                        break
                if ROOT_DIR is not None:
                    break

assert ROOT_DIR is not None, f"[FATAL] Could not find competition data (test.csv). Candidates: {_ROOT_CANDIDATES}"
print(f"[OK] ROOT_DIR = {ROOT_DIR}")

TEST_DIR   = os.path.join(ROOT_DIR, "test_images")
OUTPUT_DIR = "/kaggle/working/submission_masks"
ZIP_PATH   = "/kaggle/working/submission.zip"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---- Weight directories (discover) ----
WEIGHT_DIRS = []
_input_dir = "/kaggle/input"
if os.path.isdir(_input_dir):
    for d in os.listdir(_input_dir):
        fp = os.path.join(_input_dir, d)
        if os.path.isdir(fp) and d != os.path.basename(ROOT_DIR):
            try:
                if any(f.endswith(".pt") for f in os.listdir(fp)):
                    WEIGHT_DIRS.append(fp)
            except Exception:
                pass

    comp_dir = os.path.join(_input_dir, "competitions")
    if os.path.isdir(comp_dir):
        for d in os.listdir(comp_dir):
            fp = os.path.join(comp_dir, d)
            if os.path.isdir(fp):
                try:
                    if any(f.endswith(".pt") for f in os.listdir(fp)):
                        WEIGHT_DIRS.append(fp)
                except Exception:
                    pass

for _wd in [
    "/kaggle/input/vesuvius-nnunet-weights",
    "/kaggle/input/vesuvius-nnunet-model-a",
    "/kaggle/input/vesuvius-model-a-weights",
    "/kaggle/input/vesuvius-trained-weights-new",
]:
    if os.path.isdir(_wd) and _wd not in WEIGHT_DIRS:
        WEIGHT_DIRS.append(_wd)

WEIGHT_DIRS = sorted(list(dict.fromkeys(WEIGHT_DIRS)))
print(f"[OK] Candidate weight dirs: {WEIGHT_DIRS}")

def list_pt_files(d: str):
    try:
        return sorted([f for f in os.listdir(d) if f.endswith(".pt")])
    except Exception:
        return []

for _wd in WEIGHT_DIRS:
    pts = list_pt_files(_wd)
    if pts:
        print(f"     {_wd}: {pts[:8]}{' ...' if len(pts)>8 else ''}")

# ---- Architecture (MUST match training) ----
NUM_CLASSES       = 2
FEATURES          = [32, 64, 128, 256, 320, 320]
BLOCKS_PER_STAGE  = [1, 3, 4, 6, 6, 6]

STRIDES_ISO   = [[1,1,1], [2,2,2], [2,2,2], [2,2,2], [2,2,2], [2,2,2]]
STRIDES_ANISO = [[1,1,1], [1,2,2], [2,2,2], [2,2,2], [2,2,2], [2,2,2]]

ROI_ISO   = (192, 192, 192)  # Model A & B
ROI_ANISO = (80, 192, 192)   # Model C

# ---- Inference controls ----
INF_OVERLAP   = 0.50
TTA_ENABLED   = True

# ============================================================
# SCORE-SAFE CONTROLS (Topo/VOI focused)
# ============================================================
USE_ADAPTIVE_ENSEMBLE = False   # IMPORTANT: disable adaptive fusion (can amplify merges)
PP_T_HIGH         = 0.75
PP_T_LOW          = 0.50

# Disable bridge-creating ops by default:
PP_SMOOTH_SIGMA   = 0.0         # was 0.3 (risk)
PP_Z_RADIUS       = 0
PP_XY_RADIUS      = 0           # was 1 (risk)

# Dust removal: conservative (don’t delete thin true surface)
PP_DUST_MIN       = 200         # was 500 (safer to be lower)

PP_HOLE_FILL_MAX  = 0
PP_HOLE_FILL_VLIM = 100_000_000

# MUST EXIST (prevents NameError)
PP_REL_SMALL_FRAC = 0.01

# ---- Budget / hangtime knobs ----
MAX_SEC_PER_VOL = 240
DEBUG_VIZ       = False

print("[OK] Configuration loaded (Topo/VOI score-safe)")
hard_budget_check("Config done", min_remaining_min=10)
HB.beat("config", "loaded")

In [ ]:
# ============================================================
# CELL 3: Test Volume Discovery + Budget Sanity
# ============================================================
test_csv_path = os.path.join(ROOT_DIR, "test.csv")
assert os.path.exists(test_csv_path), f"[FATAL] test.csv not found at {test_csv_path}"

test_df = pd.read_csv(test_csv_path)
test_ids = list(dict.fromkeys(test_df["id"].astype(str).tolist()))
print(f"[OK] test.csv: {len(test_ids)} unique test volumes  (sample: {test_ids[:3]})")

# Verify volumes exist
missing = [v for v in test_ids if not os.path.exists(os.path.join(TEST_DIR, f"{v}.tif"))]
assert not missing, f"[FATAL] Missing volumes: {missing[:5]} (showing first 5)"
print(f"[OK] All {len(test_ids)} test volumes found on disk")

n_test = len(test_ids)

# ---- Budget sanity (rough) ----
# NOTE: MAX_SEC_PER_VOL should be interpreted as "per volume, per model, no TTA".
base_est_h = (n_test * MAX_SEC_PER_VOL) / 3600.0

# Conservative multipliers: 3 models + TTA(2x if flips) + overlap overhead (~1.2–1.6)
n_models = 3
tta_mult = 2.0 if TTA_ENABLED else 1.0
overlap_mult = 1.35 if INF_OVERLAP >= 0.5 else 1.20

worst_est_h = base_est_h * n_models * tta_mult * overlap_mult
best_est_h  = base_est_h * n_models * (1.0) * (1.15)  # optimistic-ish

print("\n[BUDGET] Rough inference estimate:")
print(f"  base (1 model, no TTA): {base_est_h:.2f} h")
print(f"  optimistic (3 models, mild overhead): ~{best_est_h:.2f} h")
print(f"  conservative (3 models, TTA, overlap): ~{worst_est_h:.2f} h")
print(f"[BUDGET] Remaining wall time: {remaining_h():.2f} h")

hard_budget_check("After dataset discovery", min_remaining_min=10)
log_resources("After dataset discovery")
HB.progress()
HB.beat("dataset", f"n_test={n_test}")

# Auto-safety: if conservative estimate exceeds remaining time, disable TTA early (better than Kaggle kill).
if worst_est_h > remaining_h() - (10/60):  # keep ~10 min buffer
    if TTA_ENABLED:
        print("[SAFETY] Conservative estimate exceeds remaining time budget → disabling TTA to avoid timeout.")
        TTA_ENABLED = False
        HB.progress()

In [ ]:
# ============================================================
# CELL 4: ResidualEncoderUNet Architecture (matches training exactly)
# ============================================================

class ResidualBlock3D(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv3d(channels, channels, 3, padding=1, bias=False)
        self.norm1 = nn.InstanceNorm3d(channels, eps=1e-5, affine=True)
        self.conv2 = nn.Conv3d(channels, channels, 3, padding=1, bias=False)
        self.norm2 = nn.InstanceNorm3d(channels, eps=1e-5, affine=True)
        self.act = nn.LeakyReLU(0.01, inplace=True)

    def forward(self, x):
        residual = x
        x = self.act(self.norm1(self.conv1(x)))
        x = self.norm2(self.conv2(x))
        return self.act(x + residual)


class EncoderStage(nn.Module):
    def __init__(self, in_ch, out_ch, n_blocks, stride=(1, 1, 1)):
        super().__init__()
        self.initial = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, stride=list(stride), padding=1, bias=False),
            nn.InstanceNorm3d(out_ch, eps=1e-5, affine=True),
            nn.LeakyReLU(0.01, inplace=True),
        )
        self.blocks = nn.Sequential(*[ResidualBlock3D(out_ch) for _ in range(n_blocks)])

    def forward(self, x):
        return self.blocks(self.initial(x))


class DecoderStage(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, upsample_stride=(2, 2, 2)):
        super().__init__()
        ks = list(upsample_stride)
        self.upsample = nn.ConvTranspose3d(in_ch, in_ch, kernel_size=ks, stride=ks, bias=False)
        self.conv = nn.Sequential(
            nn.Conv3d(in_ch + skip_ch, out_ch, 3, padding=1, bias=False),
            nn.InstanceNorm3d(out_ch, eps=1e-5, affine=True),
            nn.LeakyReLU(0.01, inplace=True),
        )

    def forward(self, x, skip):
        x = self.upsample(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(x, size=skip.shape[2:], mode="trilinear", align_corners=False)
        return self.conv(torch.cat([x, skip], dim=1))


class ResidualEncoderUNet(nn.Module):
    def __init__(
        self,
        in_ch=1,
        num_classes=2,
        features=(32, 64, 128, 256, 320, 320),
        blocks_per_stage=(1, 3, 4, 6, 6, 6),
        strides=((1,1,1), (2,2,2), (2,2,2), (2,2,2), (2,2,2), (2,2,2)),
        grad_ckpt_stages=None,  # present to match training signature; not used in inference
    ):
        super().__init__()
        self.grad_ckpt_stages = set(grad_ckpt_stages or [])
        n_stages = len(features)

        self.encoders = nn.ModuleList()
        for i in range(n_stages):
            in_c = in_ch if i == 0 else features[i - 1]
            self.encoders.append(EncoderStage(in_c, features[i], blocks_per_stage[i], stride=strides[i]))

        self.decoders = nn.ModuleList()
        for i in range(n_stages - 2, -1, -1):
            upsample_stride = tuple(strides[i + 1])
            self.decoders.append(
                DecoderStage(features[i + 1], features[i], features[i], upsample_stride=upsample_stride)
            )

        self.seg_heads = nn.ModuleList()
        for i in range(n_stages - 1):
            self.seg_heads.append(nn.Conv3d(features[i], num_classes, 1))

    def forward(self, x):
        # Safety: we expect N,C,Z,Y,X
        assert x.ndim == 5, f"Expected 5D input (N,C,Z,Y,X), got {x.shape}"
        skips = []
        for enc in self.encoders:
            x = enc(x)
            skips.append(x)

        x = skips[-1]
        for i, dec in enumerate(self.decoders):
            x = dec(x, skips[len(skips) - 2 - i])

        return self.seg_heads[0](x)

# Param counts
n_params_iso = sum(
    p.numel() for p in ResidualEncoderUNet(
        features=FEATURES, blocks_per_stage=BLOCKS_PER_STAGE, strides=[tuple(s) for s in STRIDES_ISO]
    ).parameters()
) / 1e6
print(f"[OK] ResidualEncoderUNet (iso): {n_params_iso:.2f}M parameters")

n_params_aniso = sum(
    p.numel() for p in ResidualEncoderUNet(
        features=FEATURES, blocks_per_stage=BLOCKS_PER_STAGE, strides=[tuple(s) for s in STRIDES_ANISO]
    ).parameters()
) / 1e6
print(f"[OK] ResidualEncoderUNet (aniso): {n_params_aniso:.2f}M parameters")

log_resources("After arch definition")
hard_budget_check("Architecture defined", min_remaining_min=10)
HB.progress()

# Optional quick dry-run (cheap, catches shape/dtype issues early)
try:
    m = ResidualEncoderUNet(features=FEATURES, blocks_per_stage=BLOCKS_PER_STAGE, strides=[tuple(s) for s in STRIDES_ISO]).to(DEVICE).eval()
    with torch.no_grad():
        x = torch.zeros((1, 1, 16, 64, 64), device=DEVICE)  # small tensor
        with autocast(device_type="cuda" if DEVICE.type=="cuda" else "cpu", dtype=AMP_DTYPE, enabled=(DEVICE.type=="cuda")):
            y = m(x)
        assert y.shape[0] == 1 and y.shape[1] == NUM_CLASSES, f"Bad output shape: {y.shape}"
    del m, x, y
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    print("[OK] Arch dry-run passed")
except Exception as e:
    print(f"[WARN] Arch dry-run skipped/failed: {e}")

In [ ]:
# ============================================================
# CELL 5 (REPLACEMENT): Model Loading + Meta + Config-Driven Ensemble Setup (S-tier)
# ============================================================
import json
import numpy as np
import torch

REQUIRE_ALL_MODELS = True  # set False only if you want auto-degrade to 2/1 models

def load_meta(meta_path: str) -> dict:
    if (meta_path is None) or (not os.path.exists(meta_path)):
        return {}
    with open(meta_path, "r") as f:
        return json.load(f)

def find_checkpoint(filename: str) -> str | None:
    for wdir in WEIGHT_DIRS:
        fp = os.path.join(wdir, filename)
        if os.path.exists(fp):
            return fp
    return None

def load_ckpt_any(weight_path: str):
    """Returns (state_dict, ckpt_dict_or_none). Supports wrapped or raw state_dict."""
    obj = torch.load(weight_path, map_location="cpu", weights_only=False)
    if isinstance(obj, dict) and "state_dict" in obj and isinstance(obj["state_dict"], dict):
        return obj["state_dict"], obj
    if isinstance(obj, dict) and all(hasattr(v, "shape") for v in obj.values()):
        return obj, None  # raw state_dict only
    raise ValueError(f"[FATAL] Unrecognized checkpoint format: {weight_path}")

def instantiate_model_from_config(cfg: dict):
    # Safe defaults fall back to notebook constants if config missing
    num_classes = int(cfg.get("num_classes", NUM_CLASSES))
    features = cfg.get("features", FEATURES)
    blocks = cfg.get("blocks_per_stage", BLOCKS_PER_STAGE)
    strides = cfg.get("strides", STRIDES_A if "STRIDES_A" in globals() else [(1,1,1)]*6)

    # Normalize strides into tuples
    strides = [tuple(s) for s in strides]

    model = ResidualEncoderUNet(
        in_ch=1,
        num_classes=num_classes,
        features=features,
        blocks_per_stage=blocks,
        strides=strides,
    )
    return model, strides, features, blocks

def load_model(weight_path: str):
    print(f"[LOAD] {weight_path}")
    sd, ckpt = load_ckpt_any(weight_path)
    cfg = {}
    if isinstance(ckpt, dict):
        cfg = ckpt.get("config", {}) if isinstance(ckpt.get("config", {}), dict) else {}
    model, strides, features, blocks = instantiate_model_from_config(cfg)
    model.load_state_dict(sd, strict=True)
    model.to(DEVICE).eval()

    # Param-count fingerprint (catches silent mismatch across versions)
    nparams = sum(p.numel() for p in model.parameters())
    print(f"  -> params={nparams:,}  strides={strides}  features={features}  blocks={blocks}")
    return model, {"config": cfg, "nparams": int(nparams), "strides": strides, "features": features, "blocks": blocks}

# --- locate weights + metas ---
ckpt_a = find_checkpoint("model_a.pt")
ckpt_b = find_checkpoint("model_b.pt")
ckpt_c = find_checkpoint("model_c.pt")

meta_a = find_checkpoint("model_a_meta.json")
meta_b = find_checkpoint("model_b_meta.json")
meta_c = find_checkpoint("model_c_meta.json")

if REQUIRE_ALL_MODELS:
    assert ckpt_a and ckpt_b and ckpt_c, f"[FATAL] missing ckpts: A={ckpt_a} B={ckpt_b} C={ckpt_c}"
    assert meta_a and meta_b and meta_c, f"[FATAL] missing metas: A={meta_a} B={meta_b} C={meta_c}"

MA = load_meta(meta_a)
MB = load_meta(meta_b)
MC = load_meta(meta_c)

# --- base weights (protect Model C; it’s your break-rescue specialist) ---
BASE_WA = 1.00
BASE_WB = 1.00
BASE_WC = 1.35

models = []
model_names = []
model_infos = []

if ckpt_a:
    m, info = load_model(ckpt_a)
    models.append(m); model_names.append("A"); model_infos.append(info)
if ckpt_b:
    m, info = load_model(ckpt_b)
    models.append(m); model_names.append("B"); model_infos.append(info)
if ckpt_c:
    m, info = load_model(ckpt_c)
    models.append(m); model_names.append("C"); model_infos.append(info)

assert len(models) >= 1, "[FATAL] No models loaded."

# Align base weights to loaded models order
name_to_w = {"A": BASE_WA, "B": BASE_WB, "C": BASE_WC}
base_ws = np.array([name_to_w[n] for n in model_names], dtype=np.float32)

print("[META] A:", MA.get("created_utc", None), "B:", MB.get("created_utc", None), "C:", MC.get("created_utc", None))
print("[OK] Models loaded:", model_names, " base_ws=", base_ws.tolist())

hard_budget_check("Models loaded", min_remaining_min=10)
HB.progress()

In [ ]:
# ============================================================
# CELL 6 (REPLACEMENT): S-tier ensemble weighting (logit fusion)
#   - voxelwise entropy confidence
#   - per-model reliability (from meta if available)
#   - poison / collapse detection
#   - Model C specialist gating on AB uncertainty + disagreement
# ============================================================
import numpy as np

EPS = 1e-8

def _sigmoid(x):
    # stable sigmoid
    x = np.clip(x, -40.0, 40.0)
    return 1.0 / (1.0 + np.exp(-x))

def _entropy_binary_from_logits(z):
    """
    Binary entropy H(p) with p = sigmoid(z). Range [0, ln2].
    Returns entropy map and confidence map in [0,1].
    """
    p = _sigmoid(z.astype(np.float32))
    p = np.clip(p, EPS, 1.0 - EPS)
    H = -(p * np.log(p) + (1.0 - p) * np.log(1.0 - p))  # [0, ln2]
    conf = 1.0 - (H / np.log(2.0))                      # [0, 1]
    return H, conf

def _safe_meta_reliability(meta: dict) -> float:
    """
    Tries to read a model reliability scalar from meta.json.
    Falls back to 1.0 if nothing is found.
    """
    if not isinstance(meta, dict) or not meta:
        return 1.0

    # Common keys you might have across your notebooks / logs
    for k in ["best_mean", "val_mean", "mean_score", "val_score", "score", "best_score"]:
        v = meta.get(k, None)
        if isinstance(v, (int, float)) and np.isfinite(v):
            # map typical 0.45-0.65-ish range to gentle multiplier ~[0.9,1.1]
            # clamp hard so you never accidentally nuke a model
            v = float(v)
            m = 1.0 + 0.8 * (v - 0.56)   # centered around ~0.56
            return float(np.clip(m, 0.85, 1.20))

    return 1.0

def _poison_check(z, name="?", verbose=False):
    """
    Detects catastrophic model outputs for a single volume:
      - NaNs/Infs
      - near-constant probabilities (collapsed)
    Returns a multiplier in [0,1] (0 = drop model).
    """
    if not np.isfinite(z).all():
        if verbose: print(f"[POISON] {name}: non-finite logits -> dropped")
        return 0.0

    p = _sigmoid(z)
    p_std = float(np.std(p))
    p_mean = float(np.mean(p))

    # near-constant prediction (collapse): tiny std
    if p_std < 1e-4:
        if verbose: print(f"[POISON] {name}: collapsed (std={p_std:.2e}, mean={p_mean:.3f}) -> dropped")
        return 0.0

    # extreme saturation everywhere (also suspicious)
    if p_mean < 1e-4 or p_mean > 1.0 - 1e-4:
        if verbose: print(f"[WARN] {name}: saturated (mean={p_mean:.6f}) -> downweighted")
        return 0.35

    return 1.0

def fuse_logits_s_tier(
    logits_list,
    model_names,
    base_ws,
    metas_by_name=None,
    *,
    alpha_conf=2.0,         # confidence exponent; higher -> stronger gating by confidence
    gamma_ab_unc=2.0,       # AB uncertainty exponent for C boost
    delta_ab_dis=2.0,       # AB disagreement exponent for C boost
    c_boost_max=2.0,        # max multiplicative boost applied to Model C locally
    verbose=False,
):
    """
    logits_list: list of numpy arrays (each is a logit volume, same shape)
    model_names: list of strings, e.g. ["A","B","C"]
    base_ws: np.array shape (M,) base weights aligned with logits_list order
    metas_by_name: dict like {"A": MA, "B": MB, "C": MC} (optional)

    Returns fused logits (same shape as each logit volume).
    """
    M = len(logits_list)
    assert M == len(model_names) == len(base_ws), "[FATAL] fuse_logits_s_tier: length mismatch"

    metas_by_name = metas_by_name or {}

    # Stack to (M, ...)
    Z = np.stack([z.astype(np.float32) for z in logits_list], axis=0)

    # Global weights = base * reliability(meta) * poison(volume)
    rel = np.ones((M,), dtype=np.float32)
    poison = np.ones((M,), dtype=np.float32)

    for i, nm in enumerate(model_names):
        rel[i] = _safe_meta_reliability(metas_by_name.get(nm, {}))
        poison[i] = _poison_check(Z[i], name=nm, verbose=verbose)

    global_w = (base_ws.astype(np.float32) * rel * poison).astype(np.float32)

    if verbose:
        print("[FUSE] global_w:", {nm: float(global_w[i]) for i, nm in enumerate(model_names)})

    # If everything died (shouldn’t), fall back to equal weights
    if float(np.sum(global_w)) <= 0:
        if verbose: print("[FUSE] all weights zero -> fallback to equal")
        global_w = np.ones_like(global_w)

    # Voxelwise confidence weights per model
    conf = []
    for i in range(M):
        _, c = _entropy_binary_from_logits(Z[i])
        conf.append(c)
    conf = np.stack(conf, axis=0)  # (M, ...)

    # Base voxelwise weights: global_w * conf^alpha
    W = global_w[:, None, None, None] * np.power(conf + 1e-6, alpha_conf)

    # --- Model C specialist gating ---
    # Boost C where A/B are uncertain OR disagree.
    # Works even if you only have 1 or 2 models (it just won’t apply).
    if "C" in model_names and M >= 2:
        idx_c = model_names.index("C")

        # pick two "non-C" models for AB proxy (prefer A and B if present)
        nonc = [i for i, nm in enumerate(model_names) if nm != "C"]
        if len(nonc) >= 2:
            i1, i2 = nonc[0], nonc[1]
        else:
            i1, i2 = nonc[0], nonc[0]

        # AB uncertainty: 1 - mean(conf)
        ab_conf_mean = 0.5 * (conf[i1] + conf[i2])
        ab_unc = 1.0 - ab_conf_mean
        ab_unc = np.clip(ab_unc, 0.0, 1.0)

        # AB disagreement: |sigmoid(z1)-sigmoid(z2)|
        p1 = _sigmoid(Z[i1])
        p2 = _sigmoid(Z[i2])
        ab_dis = np.abs(p1 - p2)
        ab_dis = np.clip(ab_dis, 0.0, 1.0)

        # gate in [1, 1+c_boost_max]
        boost = 1.0 + c_boost_max * (np.power(ab_unc, gamma_ab_unc) * np.power(ab_dis, delta_ab_dis))
        W[idx_c] *= boost.astype(np.float32)

        if verbose:
            print(f"[FUSE] Model C boost: mean={float(np.mean(boost)):.3f} max={float(np.max(boost)):.3f}")

    # Normalize weights voxelwise
    Wsum = np.sum(W, axis=0) + 1e-6
    Wn = W / Wsum

    # Fused logits
    fused = np.sum(Wn * Z, axis=0).astype(np.float32)
    return fused

# Convenience wrapper that matches your notebook globals
METAS_BY_NAME = {"A": MA if "MA" in globals() else {},
                 "B": MB if "MB" in globals() else {},
                 "C": MC if "MC" in globals() else {}}

print("[OK] S-tier fusion ready. Use:")
print("fused_logits = fuse_logits_s_tier(ensemble_logits, model_names, base_ws, METAS_BY_NAME)")
HB.progress()

In [ ]:
# ============================================================
# CELL 7: Post-Processing — Adaptive Hysteresis via FG Targets (topology-safe)
# ============================================================
import numpy as np
import scipy.ndimage as ndi
from skimage.morphology import remove_small_objects

def hysteresis_bin(p, tl, th):
    strong = p >= th
    weak = p >= tl
    lbl, n = ndi.label(weak)
    if n == 0:
        return np.zeros_like(p, dtype=np.uint8)
    strong_ids = np.unique(lbl[strong])
    strong_ids = strong_ids[strong_ids != 0]
    keep = np.isin(lbl, strong_ids)
    return keep.astype(np.uint8)

def fg_frac(mask, valid=None):
    if valid is None:
        return float(mask.mean())
    denom = float(valid.sum() + 1e-12)
    return float((mask & valid).sum() / denom)

# Use Model A fg targets as canonical (trained on same folds)
TARGET_FG_MED = float(MA.get("target_fg_med", 0.0))
TARGET_FG_LO  = float(MA.get("target_fg_lo", 0.0))
TARGET_FG_HI  = float(MA.get("target_fg_hi", 1.0))
TH_PACK = MA.get("th_pack", [{"tl":0.55,"th":0.85}])

def pick_best_threshold_pack(prob, valid=None):
    """
    Choose tl/th that yields fg fraction closest to TARGET_FG_MED,
    but constrained to [TARGET_FG_LO, TARGET_FG_HI] if possible.
    """
    best = None
    for item in TH_PACK:
        tl, th = float(item["tl"]), float(item["th"])
        pred = hysteresis_bin(prob, tl, th).astype(bool)
        frac = fg_frac(pred, valid)
        # penalty: distance to median + extra penalty if outside IQR
        penalty = abs(frac - TARGET_FG_MED)
        if frac < TARGET_FG_LO: penalty += (TARGET_FG_LO - frac) * 2.0
        if frac > TARGET_FG_HI: penalty += (frac - TARGET_FG_HI) * 2.0
        if (best is None) or (penalty < best[0]):
            best = (penalty, tl, th, frac)
    return best[1], best[2], best[3]

def topo_safe_cleanup(mask_u8, min_size=48):
    # remove tiny specks only (too aggressive hurts topo)
    m = mask_u8.astype(bool)
    m = remove_small_objects(m, min_size=min_size)
    return m.astype(np.uint8)

def topo_postprocess(
    prob: np.ndarray,
    t_low: float,
    t_high: float,
    z_radius: int = 0,
    xy_radius: int = 0,
    dust_min: int = 200,
    hole_fill_max: int = 0,
    hole_fill_vlim: int = 0,
    rel_small_frac: float = 0.01,
):
    """
    TOPO/VOI-safe postproc:
      - hysteresis (keeps connectivity without thickening too much)
      - remove tiny specks only
      - intentionally NO closing/dilation/smoothing by default (bridge risk)
    """
    m = hysteresis_bin(prob, float(t_low), float(t_high)).astype(np.uint8)

    # Conservative dust: absolute minimum + relative minimum
    # (relative helps across varying volume sizes)
    rel_min = int(rel_small_frac * m.size)
    min_keep = int(max(dust_min, rel_min, 1))
    m = topo_safe_cleanup(m, min_size=min_keep)

    return m.astype(np.uint8, copy=False)

In [ ]:
# ============================================================
# CELL 8: Smoke Test (fast + informative + OOM-aware)
# ============================================================
print("[SMOKE] Starting mini end-to-end smoke test...")
_smoke_ok = False

_smoke_vars = {}  # track created vars for safe cleanup

try:
    hard_budget_check("Smoke start", min_remaining_min=10)
    log_resources("Smoke start")

    _smoke_id = test_ids[0]
    _smoke_path = os.path.join(TEST_DIR, f"{_smoke_id}.tif")

    HB.beat("smoke", f"load {_smoke_id}")
    _smoke_vol = load_test_volume(_smoke_path)
    _smoke_vars["_smoke_vol"] = _smoke_vol
    print(f"  Volume loaded: id={_smoke_id} shape={_smoke_vol.shape}, dtype={_smoke_vol.dtype}")

    D, H, W = _smoke_vol.shape
    # Fast ROI: shallow Z (so it runs fast), reasonable XY
    _roi = (min(D, 32), min(H, 64), min(W, 64))
    print(f"  Smoke ROI={_roi}, overlap=0.15, TTA=OFF")

    # Quick per-model forward sanity (single patch each)
    for mi, m in enumerate(models):
        HB.beat("smoke", f"model{mi} patch")
        patch = _smoke_vol[:_roi[0], :_roi[1], :_roi[2]]
        inp = torch.from_numpy(patch[None, None]).to(DEVICE)
        try:
            with torch.no_grad():
                with _autocast_ctx():
                    out = m(inp)
            out = out.float().cpu()
            print(f"  Model{mi} OK: out={tuple(out.shape)} range=[{out.min():.2f},{out.max():.2f}]")
        except Exception as e:
            if is_cuda_oom(e):
                oom_failfast(f"smoke model{mi} forward", e)
            raise
        finally:
            del inp, out
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()
        HB.progress()

    # Minimal inference using first model only (fast path)
    HB.beat("smoke", "predict")
    _smoke_logits = predict_with_tta(
        models[0],
        _smoke_vol,
        roi_size=_roi,
        overlap=0.15,
        num_classes=NUM_CLASSES,
        tta_mode=False,
        stage="smoke_pred",
    )
    _smoke_vars["_smoke_logits"] = _smoke_logits
    print(f"  Logits: shape={_smoke_logits.shape}, range=[{_smoke_logits.min():.2f}, {_smoke_logits.max():.2f}]")

    _smoke_probs = np_softmax(_smoke_logits, axis=-1)[..., 1].astype(np.float32)
    _smoke_vars["_smoke_probs"] = _smoke_probs

    HB.beat("smoke", "postproc")
    _smoke_mask = topo_postprocess(
        _smoke_probs,
        t_low=PP_T_LOW, t_high=PP_T_HIGH,
        z_radius=PP_Z_RADIUS, xy_radius=PP_XY_RADIUS,
        dust_min=PP_DUST_MIN,
        hole_fill_max=PP_HOLE_FILL_MAX,
        hole_fill_vlim=PP_HOLE_FILL_VLIM,
    )
    _smoke_vars["_smoke_mask"] = _smoke_mask

    fg_pct = 100.0 * float(_smoke_mask.mean())
    print(f"  Mask: shape={_smoke_mask.shape}, dtype={_smoke_mask.dtype}, "
          f"unique={np.unique(_smoke_mask).tolist()}, FG={fg_pct:.4f}%")

    assert _smoke_mask.shape == _smoke_vol.shape, "Shape mismatch!"
    assert _smoke_mask.dtype == np.uint8
    assert set(np.unique(_smoke_mask)).issubset({0, 1})

    # Write -> ZIP -> re-read round trip
    HB.beat("smoke", "zip")
    _stif = os.path.join(OUTPUT_DIR, f"_smoke_{_smoke_id}.tif")
    _szip = os.path.join(OUTPUT_DIR, "_smoke.zip")
    tifffile.imwrite(_stif, _smoke_mask)

    with zipfile.ZipFile(_szip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        zf.write(_stif, arcname=f"{_smoke_id}.tif")

    with zipfile.ZipFile(_szip, "r") as zf:
        with zf.open(f"{_smoke_id}.tif") as sf:
            _re = tifffile.imread(sf)
            assert _re.dtype == np.uint8 and set(np.unique(_re)).issubset({0, 1})

    os.remove(_stif); os.remove(_szip)

    _smoke_ok = True
    print(f"[SMOKE] PASSED — full chain verified for {_smoke_id}")

except Exception as e:
    print(f"[SMOKE] FAILED: {e}")
    import traceback; traceback.print_exc()

finally:
    # Safe cleanup
    for k in list(_smoke_vars.keys()):
        try:
            del _smoke_vars[k]
        except Exception:
            pass
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    log_resources("Smoke end")

assert _smoke_ok, "[FATAL] Smoke test failed — do NOT proceed"

hard_budget_check("Smoke test done", min_remaining_min=10)
HB.progress()

In [ ]:
# ============================================================
# CELL 9 (REPLACEMENT): Main Inference Loop — proactive runtime + safe degradation (S-tier)
# ============================================================

hard_budget_check("Starting inference loop", min_remaining_min=15)
log_resources("Inference start")

degradation = DegradationState(start_level=1)  # SAFE DEFAULT START
volume_times = []

# Proactive runtime guard (tune once you see real timings)
WINDOW = 5
TARGET_SEC_PER_VOL = 180        # if slower than this on avg, degrade
HARD_MAX_SEC_PER_VOL = 420      # if a single vol takes longer than this, degrade immediately

zip_path = ZIP_PATH
print(f"[WRITE] {zip_path}")

# Optional metas dict (if present)
metas_by_name = METAS_BY_NAME if "METAS_BY_NAME" in globals() else {}

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for idx, iid in enumerate(test_ids, start=1):
        t0 = time.time()
        HB.beat("infer", f"vol {idx}/{len(test_ids)} id={iid}")

        path = os.path.join(TEST_DIR, f"{iid}.tif")
        vol = load_test_volume(path)

        # Current degradation settings
        cur = degradation.current()
        overlap = float(cur["overlap"])
        tta_mode = cur["tta"]
        roi_xy = tuple(cur["roi_xy"])

        # Keep your Z logic but override XY here
        roi = (ROI_ISO[0], roi_xy[0], roi_xy[1])

        print(f"\n[{idx}/{len(test_ids)}] id={iid} shape={vol.shape} | overlap={overlap} | TTA={tta_mode} | roi={roi}")

        # -------- Per-model inference (OOM-safe, retry only on CUDA OOM) --------
        def run_models(stage_prefix: str):
            outs = []
            for mi, m in enumerate(models):
                lg = predict_with_tta(
                    m,
                    vol,
                    roi_size=roi,
                    overlap=overlap,
                    num_classes=NUM_CLASSES,
                    tta_mode=tta_mode,
                    stage=f"{stage_prefix}_m{mi}",
                )
                outs.append(lg)
            return outs

        try:
            model_logits = run_models("infer")
        except Exception as e:
            if is_cuda_oom(e):
                print(f"[OOM] Inference failure on id={iid}: {repr(e)}")
                print("[OOM] Degrading and retrying once...")

                degradation.degrade()
                cur = degradation.current()
                overlap = float(cur["overlap"])
                tta_mode = cur["tta"]
                roi_xy = tuple(cur["roi_xy"])
                roi = (ROI_ISO[0], roi_xy[0], roi_xy[1])

                try:
                    model_logits = run_models("retry")
                except Exception as e2:
                    if is_cuda_oom(e2):
                        print(f"[OOM] Retry failed on id={iid}: {repr(e2)}")
                        print("[OOM] Degrading again and retrying once more...")

                        degradation.degrade()
                        cur = degradation.current()
                        overlap = float(cur["overlap"])
                        tta_mode = cur["tta"]
                        roi_xy = tuple(cur["roi_xy"])
                        roi = (ROI_ISO[0], roi_xy[0], roi_xy[1])

                        model_logits = run_models("retry2")
                    else:
                        raise
            else:
                raise

        # -------- Ensemble fusion (logits) [S-tier] --------
        # model_logits is a list aligned with `models` order (which matches model_names from CELL 5)
        logits_list = model_logits
        names_list = list(model_names[:len(logits_list)])

        # weights: prefer your existing ENSEMBLE_WEIGHTS if present, else fall back to base_ws
        if "ENSEMBLE_WEIGHTS" in globals() and ENSEMBLE_WEIGHTS is not None:
            if isinstance(ENSEMBLE_WEIGHTS, dict):
                w = np.array([float(ENSEMBLE_WEIGHTS.get(n, 1.0)) for n in names_list], dtype=np.float32)
            else:
                w = np.array(list(ENSEMBLE_WEIGHTS)[:len(names_list)], dtype=np.float32)
        else:
            # base_ws aligned with model_names
            w = np.array([float(base_ws[model_names.index(n)]) for n in names_list], dtype=np.float32)

        # Normalize each model logits to a single binary logit (fg - bg) so fusion is correct
        logits_list_norm = []
        for z in logits_list:
            z = z.astype(np.float32)

            # Case A: last axis is channels (..., 2)
            if z.ndim >= 4 and z.shape[-1] == 2:
                z = z[..., 1] - z[..., 0]

            # Case B: first axis is channels (2, ...)
            elif z.ndim >= 4 and z.shape[0] == 2:
                z = z[1] - z[0]

            # Else: already single-channel binary logit
            logits_list_norm.append(z)

        fused_logits = fuse_logits_s_tier(
            logits_list_norm,
            names_list,
            w,
            metas_by_name,
            verbose=False,   # True for debugging one volume
        )

        if not np.isfinite(fused_logits).all():
            raise RuntimeError(f"[FATAL] fused_logits non-finite on id={iid}")

        # -------- Prob + postproc --------
        prob = (1.0 / (1.0 + np.exp(-np.clip(fused_logits, -40.0, 40.0)))).astype(np.float32)

        pmin, pmax = float(prob.min()), float(prob.max())
        if not (0.0 <= pmin <= 1.0 and 0.0 <= pmax <= 1.0):
            raise RuntimeError(f"[FATAL] prob out of [0,1] on id={iid}: min={pmin} max={pmax}")

        # Threshold selection
        if PP_USE_DYNAMIC_THRESH:
            tl, th, frac = pick_best_threshold_pack(prob, valid=None)
        else:
            tl, th = float(PP_T_LOW), float(PP_T_HIGH)

        final_mask = topo_postprocess(
            prob,
            t_low=tl, t_high=th,
            z_radius=PP_Z_RADIUS, xy_radius=PP_XY_RADIUS,
            dust_min=PP_DUST_MIN,
            hole_fill_max=PP_HOLE_FILL_MAX,
            hole_fill_vlim=int(PP_HOLE_FILL_VLIM) if "PP_HOLE_FILL_VLIM" in globals() else 0,
            rel_small_frac=float(PP_REL_SMALL_FRAC) if "PP_REL_SMALL_FRAC" in globals() else 0.0,
        )

        # -------- Write .tif into zip --------
        out_name = f"{iid}.tif"
        with io.BytesIO() as buf:
            tifffile.imwrite(buf, final_mask.astype(np.uint8, copy=False))
            zf.writestr(out_name, buf.getvalue())

        # -------- Runtime accounting --------
        dt = time.time() - t0
        volume_times.append(dt)
        print(
            f"[DONE] id={iid} time={dt:.1f}s | mean(last{min(len(volume_times),WINDOW)})="
            f"{np.mean(volume_times[-min(len(volume_times),WINDOW):]):.1f}s"
        )

        # Hard per-volume trigger
        if dt > HARD_MAX_SEC_PER_VOL:
            print(f"[RUNTIME] {dt:.1f}s > {HARD_MAX_SEC_PER_VOL}s → degrade")
            degradation.degrade()

        # Proactive average trigger
        if len(volume_times) >= WINDOW:
            avg = float(np.mean(volume_times[-WINDOW:]))
            if avg > TARGET_SEC_PER_VOL:
                print(f"[RUNTIME] avg last {WINDOW} vols = {avg:.1f}s > {TARGET_SEC_PER_VOL}s → degrade")
                degradation.degrade()

        # Cleanup
        del vol, model_logits, logits_list, logits_list_norm, fused_logits, prob, final_mask
        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

        HB.progress()

print(f"[OK] Wrote {zip_path}")

In [ ]:
# ============================================================
# CELL 10: ZIP Validation + Health Checks (hardened)
# ============================================================
print(f"\n{'='*70}")
print("[VALIDATOR] ZIP VALIDATION")
print(f"{'='*70}")

hard_budget_check("Validator start", min_remaining_min=5)
HB.beat("validator", "start")

assert os.path.exists(ZIP_PATH), f"[FATAL] ZIP missing: {ZIP_PATH}"
zip_size_mb = os.path.getsize(ZIP_PATH) / (1024**2)
print(f"[INFO] ZIP size: {zip_size_mb:.1f} MB")

# Basic size sanity: extremely tiny zip usually means broken output
assert zip_size_mb > 1.0, f"[FATAL] ZIP size too small ({zip_size_mb:.2f} MB) — likely empty/broken."

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    zfiles = [n for n in zf.namelist() if n.endswith(".tif")]

    assert len(zfiles) == len(test_ids), f"File count mismatch: {len(zfiles)} vs {len(test_ids)}"

    nested = [n for n in zfiles if "/" in n]
    assert not nested, f"Nested paths: {nested[:5]}"

    zip_ids = sorted(n.replace(".tif", "") for n in zfiles)
    exp_ids = sorted(str(i) for i in test_ids)
    assert zip_ids == exp_ids, "ID mismatch!"

    # Member size sanity (catches zero-byte entries)
    infos = {zi.filename: zi.file_size for zi in zf.infolist()}
    zero_members = [fn for fn, sz in infos.items() if fn.endswith(".tif") and sz == 0]
    assert not zero_members, f"[FATAL] Zero-byte tif entries in zip: {zero_members[:5]}"

    # Spot-check 3 files + ensure non-trivial content loads
    check_indices = [0, len(zfiles) // 2, -1]
    for ci in check_indices:
        fn = zfiles[ci]
        with zf.open(fn) as f:
            arr = tifffile.imread(f)
            assert arr.dtype == np.uint8, f"{fn} dtype={arr.dtype}"
            u = set(np.unique(arr).tolist())
            assert u.issubset({0, 1}), f"{fn} bad values {sorted(list(u))}"
            assert arr.size > 0, f"{fn} empty array?"
    print(f"[OK] {len(zfiles)} files | root-level | uint8 {{0,1}} | IDs match | members nonzero")

print(f"[OK] {ZIP_PATH} ({zip_size_mb:.1f} MB)")

# ============================================================
# Health checks
# ============================================================
print(f"\n{'='*70}")
print("[HEALTH] Red-flags & stats")
print(f"{'='*70}")

critical = False

# If we required all models, enforce we didn't knowingly degrade
if "REQUIRE_ALL_MODELS" in globals() and REQUIRE_ALL_MODELS:
    # If you kept red_flags, any degradation should have raised earlier; still check.
    if red_flags:
        print("[RED-FLAG] red_flags present even though REQUIRE_ALL_MODELS=True")
        critical = True

if red_flags:
    print(f"[RED-FLAG] {len(red_flags)} issues (first 10):")
    for r in red_flags[:10]:
        print(f"   - {r}")
    critical = True

if empty_ids:
    print(f"[RED-FLAG] {len(empty_ids)} empty masks (first 10): {empty_ids[:10]}")
    critical = True

if volume_times:
    avg_t = float(np.mean(volume_times))
    max_t = float(np.max(volume_times))
    tot_m = float(np.sum(volume_times) / 60.0)
    print(f"[STATS] Time: avg={avg_t:.1f}s  max={max_t:.1f}s  total={tot_m:.1f} min")

    # Runtime creep / fragmentation heuristic
    if len(volume_times) > 5:
        f5, l5 = float(np.mean(volume_times[:5])), float(np.mean(volume_times[-5:]))
        if l5 > f5 * 1.5:
            print(f"[RED-FLAG] Runtime creep (possible fragmentation): first5={f5:.1f}s -> last5={l5:.1f}s")
            critical = True

if fg_pcts:
    fg_avg = float(np.mean(fg_pcts))
    fg_min = float(np.min(fg_pcts))
    fg_max = float(np.max(fg_pcts))
    print(f"[STATS] FG%: avg={fg_avg:.4f}%  min={fg_min:.4f}%  max={fg_max:.4f}%")

log_resources("Validator end")
print()
if critical:
    print(f"[WARN] SUBMISSION READY WITH WARNINGS (elapsed={elapsed_h():.2f} h)")
else:
    print(f"[OK] SUBMISSION READY — ALL VALIDATORS PASSED (elapsed={elapsed_h():.2f} h)")
HB.progress()